In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from skimage import data
from skimage.feature import graycomatrix, graycoprops
from mpl_toolkits.mplot3d import Axes3D

# Load grayscale image
img = data.camera()

# Select different texture patches
patch_size = 48

patch_locations = {
    "Sky": (20, 20),
    "Building": (400, 100),
    "Coat": (220, 350)
}

patches = {}

for name, (x, y) in patch_locations.items():
    patches[name] = img[y:y+patch_size, x:x+patch_size]

# Display original image with selected patches
plt.figure(figsize=(8, 6))
plt.imshow(img, cmap="gray")
plt.title("Original Image with Selected Patches")

colors = ["red", "lime", "blue"]

for (name, (x, y)), color in zip(patch_locations.items(), colors):
    rect = Rectangle(
        (x, y), patch_size, patch_size,
        linewidth=2, edgecolor=color, facecolor="none"
    )
    plt.gca().add_patch(rect)
    plt.text(x, y-5, name, color=color, fontsize=10)

plt.axis("off")
plt.show()

# Calculate GLCM features for each patch
features = {}

for name, patch in patches.items():

    # Reduce gray levels from 256 to 16
    quantized = (patch // 16).astype(np.uint8)

    glcm = graycomatrix(
        quantized,
        distances=[1],
        angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
        levels=16,
        symmetric=True,
        normed=True
    )

    features[name] = {
        "Dissimilarity": np.mean(graycoprops(glcm, "dissimilarity")),
        "Correlation": np.mean(graycoprops(glcm, "correlation")),
        "Contrast": np.mean(graycoprops(glcm, "contrast")),
        "Homogeneity": np.mean(graycoprops(glcm, "homogeneity")),
        "Energy": np.mean(graycoprops(glcm, "energy"))
    }

# Display the extracted features
print("GLCM Texture Features\n")

for name, values in features.items():
    print(name)
    for feature, value in values.items():
        print(f"{feature}: {value:.4f}")
    print()

# Display selected patches side-by-side
plt.figure(figsize=(12, 4))

for i, (name, patch) in enumerate(patches.items()):
    plt.subplot(1, 3, i+1)
    plt.imshow(patch, cmap="gray")
    plt.title(name)
    plt.axis("off")

plt.tight_layout()
plt.show()

# Scatter plot: Dissimilarity vs Correlation
plt.figure(figsize=(7, 5))

for name, values in features.items():
    plt.scatter(
        values["Dissimilarity"],
        values["Correlation"],
        s=100,
        label=name
    )
    plt.annotate(
        name,
        (values["Dissimilarity"], values["Correlation"])
    )

plt.xlabel("Dissimilarity")
plt.ylabel("Correlation")
plt.title("GLCM Feature Comparison")
plt.legend()
plt.grid(True)
plt.show()

# 3D representation of feature space
fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(111, projection="3d")

for name, values in features.items():
    ax.scatter(
        values["Dissimilarity"],
        values["Correlation"],
        values["Contrast"],
        s=100,
        label=name
    )
    ax.text(
        values["Dissimilarity"],
        values["Correlation"],
        values["Contrast"],
        name
    )

ax.set_xlabel("Dissimilarity")
ax.set_ylabel("Correlation")
ax.set_zlabel("Contrast")
ax.set_title("3D GLCM Feature Space")
ax.legend()

plt.show()